# Mini Project AI & Data Engineer
## Data Barang yang Diimpor Indonesia dari China

**Sumber:** UN Comtrade API  
**Cakupan:** Indonesia sebagai reporter, China sebagai partner, arus impor, data tahunan 2024, komoditas HS 6 digit.

### Tahapan proyek
1. Mendapatkan dan menyimpan API key
2. Menguji satu panggilan API
3. Membungkus akses API dalam class (OOP)
4. Mengubah respons JSON menjadi DataFrame
5. Membersihkan nilai kosong, duplikat, dan tipe data
6. Membuat ringkasan sederhana
7. Menyimpan dataset akhir ke CSV

Notebook menggunakan satu sumber data dan menargetkan minimal 100 baris bersih.

# Bagian 1 — Mendapatkan API Key

1. Buat akun UN Comtrade di [comtradeplus.un.org](https://comtradeplus.un.org/).
2. Masuk ke [UN Comtrade Developer Portal](https://comtradedeveloper.un.org/).
3. Pilih produk API gratis/trial yang tersedia dan salin **subscription key**.
4. Buat file `.env` di folder yang sama dengan notebook:

```env
COMTRADE_API_KEY=isi_subscription_key_anda
```

Jangan menulis key langsung di notebook atau mengunggah `.env` ke GitHub.

In [ ]:
# Jalankan sekali jika library belum tersedia
%pip install requests pandas python-dotenv --quiet

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("COMTRADE_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ditemukan. Periksa file .env dan nama COMTRADE_API_KEY.")

# Bagian 2 — Memahami Parameter API

Endpoint final data:

```text
https://comtradeapi.un.org/data/v1/get/C/A/HS
```

| Parameter | Nilai | Arti |
|---|---:|---|
| `reporterCode` | 360 | Indonesia sebagai negara pelapor |
| `partnerCode` | 156 | China sebagai negara mitra |
| `flowCode` | M | Arus impor |
| `period` | 2024 | Tahun data |
| `cmdCode` | AG6 | Semua komoditas agregasi HS 6 digit |
| `maxRecords` | 500 | Maksimum baris yang diminta |
| `includeDesc` | true | Sertakan nama/deskripsi |

`C/A/HS` berarti Commodity / Annual / Harmonized System.

In [ ]:
ALAMAT_API = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

parameter_uji = {
    "reporterCode": "360",
    "partnerCode": "156",
    "flowCode": "M",
    "period": "2024",
    "cmdCode": "AG6",
    "maxRecords": 5,
    "includeDesc": "true",
    "subscription-key": API_KEY,
}

if not API_KEY:
    raise ValueError("API key belum tersedia. Isi COMTRADE_API_KEY pada file .env.")

response_uji = requests.get(ALAMAT_API, params=parameter_uji, timeout=60)
print("Status code:", response_uji.status_code)
response_uji.raise_for_status()
hasil_uji = response_uji.json()
print("Jumlah record diterima:", len(hasil_uji.get("data", [])))

In [ ]:
# Melihat struktur satu record tanpa menampilkan API key
data_uji = hasil_uji.get("data", [])
if not data_uji:
    raise ValueError("API berhasil dihubungi, tetapi tidak mengembalikan data.")
record_pertama = data_uji[0]
record_pertama

# Bagian 3 — Membungkus Akses API dalam Class

Class menyimpan API key dan endpoint sebagai atribut. Method `ambil_data_impor()` menerima tahun dan jumlah record, menangani gangguan sementara dengan retry, memvalidasi respons, lalu mengembalikan DataFrame.

In [ ]:
class KlienComtrade:
    """Klien sederhana untuk mengambil impor Indonesia dari China."""

    def __init__(self, api_key, timeout=60, jumlah_retry=2):
        if not api_key:
            raise ValueError("API key wajib diisi.")
        self.api_key = api_key
        self.timeout = timeout
        self.jumlah_retry = jumlah_retry
        self.alamat_api = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

    def ambil_data_impor(self, tahun=2024, maksimum_record=500):
        parameter = {
            "reporterCode": "360",
            "partnerCode": "156",
            "flowCode": "M",
            "period": str(tahun),
            "cmdCode": "AG6",
            "maxRecords": int(maksimum_record),
            "includeDesc": "true",
            "subscription-key": self.api_key,
        }

        error_terakhir = None
        for percobaan in range(1, self.jumlah_retry + 2):
            try:
                response = requests.get(
                    self.alamat_api, params=parameter, timeout=self.timeout
                )
                response.raise_for_status()
                hasil = response.json()

                if "data" not in hasil:
                    raise ValueError("Respons API tidak memiliki key 'data'.")

                return pd.DataFrame(hasil["data"])
            except (requests.RequestException, ValueError) as error:
                error_terakhir = error
                if percobaan <= self.jumlah_retry:
                    print(f"Percobaan {percobaan} gagal. Mengulang...")
                    time.sleep(2 * percobaan)

        raise RuntimeError(f"Pengambilan data gagal: {error_terakhir}")

In [ ]:
klien = KlienComtrade(API_KEY)
df_mentah = klien.ambil_data_impor(tahun=2024, maksimum_record=500)

print(f"Jumlah baris mentah: {len(df_mentah)}")
print(f"Jumlah kolom mentah: {len(df_mentah.columns)}")
df_mentah.head()

# Bagian 4 — Memilih Kolom yang Relevan

Respons API memiliki banyak kolom teknis. Proyek ini mempertahankan identitas perdagangan, kode/nama barang, kuantitas, berat bersih, dan nilai impor.

In [ ]:
PETA_KOLOM = {
    "period": "Tahun",
    "reporterCode": "Kode_Reporter",
    "reporterDesc": "Negara_Reporter",
    "partnerCode": "Kode_Mitra",
    "partnerDesc": "Negara_Mitra",
    "flowCode": "Kode_Arus",
    "flowDesc": "Arus_Perdagangan",
    "cmdCode": "Kode_HS",
    "cmdDesc": "Nama_Barang",
    "qtyUnitAbbr": "Satuan_Kuantitas",
    "qty": "Kuantitas",
    "netWgt": "Berat_Bersih_Kg",
    "primaryValue": "Nilai_Impor_USD",
    "isReported": "Dilaporkan_Langsung",
}

kolom_tersedia = [kolom for kolom in PETA_KOLOM if kolom in df_mentah.columns]
df_impor = df_mentah[kolom_tersedia].rename(columns=PETA_KOLOM).copy()
print("Kolom terpilih:", df_impor.columns.tolist())
df_impor.head()

# Bagian 5 — Pemeriksaan Awal

Tiga pemeriksaan wajib: nilai kosong, duplikat, dan tipe data. Kode HS diperlakukan sebagai teks agar angka nol di depan tidak hilang.

In [ ]:
print("1. Nilai kosong per kolom:")
print(df_impor.isna().sum())
print("\n2. Duplikat berdasarkan Tahun + Kode HS:")
print(df_impor.duplicated(subset=["Tahun", "Kode_HS"]).sum())
print("\n3. Tipe data awal:")
print(df_impor.dtypes)

## Aturan Pembersihan

- Buang record tanpa `Kode_HS` atau `Nama_Barang` karena identitas barang tidak lengkap.
- Ubah `Tahun` menjadi integer nullable.
- Simpan `Kode_HS` sebagai string dan pertahankan enam digit.
- Ubah kuantitas, berat, dan nilai impor menjadi numerik.
- Nilai kosong pada ukuran numerik diisi 0 karena API dapat tidak melaporkan satuan tertentu.
- Satuan kuantitas kosong diisi `N/A`.
- Hapus duplikat berdasarkan kombinasi `Tahun` dan `Kode_HS`.
- Pastikan hanya baris Indonesia–China dengan arus impor yang tersisa.

In [ ]:
def bersihkan_teks(teks):
    if pd.isna(teks) or not str(teks).strip():
        return pd.NA
    return " ".join(str(teks).split())


def ubah_numerik(seri, isi_kosong=0):
    return pd.to_numeric(seri, errors="coerce").fillna(isi_kosong)


def bersihkan_data_impor(df):
    hasil = df.copy()

    for kolom in ["Negara_Reporter", "Negara_Mitra", "Arus_Perdagangan",
                  "Nama_Barang", "Satuan_Kuantitas"]:
        if kolom in hasil.columns:
            hasil[kolom] = hasil[kolom].apply(bersihkan_teks)

    hasil = hasil.dropna(subset=["Kode_HS", "Nama_Barang"])
    hasil["Tahun"] = pd.to_numeric(hasil["Tahun"], errors="coerce").astype("Int64")
    hasil["Kode_HS"] = hasil["Kode_HS"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)

    for kolom in ["Kuantitas", "Berat_Bersih_Kg", "Nilai_Impor_USD"]:
        if kolom in hasil.columns:
            hasil[kolom] = ubah_numerik(hasil[kolom])

    if "Satuan_Kuantitas" in hasil.columns:
        hasil["Satuan_Kuantitas"] = hasil["Satuan_Kuantitas"].fillna("N/A")
    if "Dilaporkan_Langsung" in hasil.columns:
        hasil["Dilaporkan_Langsung"] = hasil["Dilaporkan_Langsung"].astype("boolean")

    hasil = hasil[
        (hasil["Kode_Reporter"] == 360)
        & (hasil["Kode_Mitra"] == 156)
        & (hasil["Kode_Arus"] == "M")
    ]
    hasil = hasil.drop_duplicates(subset=["Tahun", "Kode_HS"], keep="first")
    hasil = hasil.sort_values("Nilai_Impor_USD", ascending=False).reset_index(drop=True)
    return hasil


df_bersih = bersihkan_data_impor(df_impor)
df_bersih.head()

In [ ]:
# Pemeriksaan akhir / quality gate
assert len(df_bersih) >= 100, "Dataset akhir belum mencapai minimal 100 baris."
assert df_bersih["Kode_HS"].notna().all(), "Masih ada Kode HS kosong."
assert df_bersih["Nama_Barang"].notna().all(), "Masih ada nama barang kosong."
assert not df_bersih.duplicated(subset=["Tahun", "Kode_HS"]).any(), "Masih ada duplikat."
assert (df_bersih["Kode_Reporter"] == 360).all()
assert (df_bersih["Kode_Mitra"] == 156).all()
assert (df_bersih["Kode_Arus"] == "M").all()

print(f"Jumlah baris akhir       : {len(df_bersih)}")
print(f"Minimal 100 baris        : {len(df_bersih) >= 100}")
print(f"Total nilai kosong       : {int(df_bersih.isna().sum().sum())}")
print(f"Jumlah duplikat          : {df_bersih.duplicated(subset=['Tahun', 'Kode_HS']).sum()}")
print("\nTipe data akhir:")
print(df_bersih.dtypes)

# Bagian 6 — Analisis Sederhana

Tampilkan 10 barang dengan nilai impor terbesar. Hasil ini dapat menjadi temuan utama presentasi.

In [ ]:
top_10 = (
    df_bersih[["Kode_HS", "Nama_Barang", "Berat_Bersih_Kg", "Nilai_Impor_USD"]]
    .nlargest(10, "Nilai_Impor_USD")
)
top_10

In [ ]:
total_nilai = df_bersih["Nilai_Impor_USD"].sum()
total_berat = df_bersih["Berat_Bersih_Kg"].sum()

print(f"Total nilai impor pada record yang diambil : USD {total_nilai:,.2f}")
print(f"Total berat bersih                      : {total_berat:,.2f} kg")
print(f"Komoditas bernilai impor terbesar       : {top_10.iloc[0]['Nama_Barang']}")

# Bagian 7 — Menyimpan Hasil

CSV disimpan dengan encoding `utf-8-sig` agar lebih nyaman dibuka di Excel 2016 dan karakter non-ASCII tetap terbaca.

In [ ]:
nama_file = "dataset_impor_indonesia_dari_china_2024.csv"
df_bersih.to_csv(nama_file, index=False, encoding="utf-8-sig")

df_cek = pd.read_csv(nama_file, dtype={"Kode_HS": str})
print(f"Data disimpan: {nama_file}")
print(f"Verifikasi: {len(df_cek)} baris dan {len(df_cek.columns)} kolom")
df_cek.head()

# Bagian 8 — Bahan Slide Presentasi

Gunakan keluaran cell berikut untuk menjelaskan sumber, teknik, kualitas, dan hasil proyek.

In [ ]:
jumlah_duplikat_mentah = df_impor.duplicated(subset=["Tahun", "Kode_HS"]).sum()
jumlah_kosong_mentah = int(df_impor.isna().sum().sum())

print("=" * 65)
print("RINGKASAN MINI PROJECT")
print("=" * 65)
print("Sumber              : UN Comtrade API")
print("Topik               : Impor barang Indonesia dari China")
print("Periode             : 2024")
print("Klasifikasi         : HS 6 digit")
print(f"Baris mentah         : {len(df_mentah)}")
print(f"Baris akhir          : {len(df_bersih)}")
print(f"Sel kosong awal      : {jumlah_kosong_mentah}")
print(f"Duplikat awal        : {jumlah_duplikat_mentah}")
print(f"Total nilai impor    : USD {total_nilai:,.2f}")
print("Class               : KlienComtrade")
print("Function            : bersihkan_teks, ubah_numerik, bersihkan_data_impor")
print("Output              : dataset_impor_indonesia_dari_china_2024.csv")
print("=" * 65)

# Kesimpulan

Notebook telah memenuhi komponen mini project:

| Kriteria | Implementasi |
|---|---|
| API menggunakan API key | UN Comtrade Final Data API + `COMTRADE_API_KEY` |
| Data sesuai topik | Impor Indonesia (360) dari China (156), flow `M` |
| OOP | Class `KlienComtrade` |
| Function/modularitas | Tiga fungsi pembersihan |
| Data tabular | JSON diubah menjadi pandas DataFrame |
| Data cleaning | Missing value, duplikat, tipe data, validasi scope |
| Minimal 100 baris | Diperiksa dengan `assert` |
| Penyimpanan | CSV UTF-8-SIG |
| Bahan presentasi | Ringkasan metrik dan 10 komoditas terbesar |

> Catatan: nilai total hanya menjumlahkan record yang dikembalikan sesuai batas query, sehingga jangan menyebutnya sebagai keseluruhan impor nasional jika API membatasi jumlah record.